In [1]:
import ee
import geopandas as gpd
from geesat import geogee 
import geesat
geesat.authenticate_gee()

### 1. Load Landsat LST and create monthly composite 

In [ ]:
tiles = gpd.read_file(r"\aoi_tiles\training_tiles.geojson")
tile = tiles.iloc[[0]]
aoi = geesat.gdf_to_ee(tile)

- **Load Landsat LST and make monthly composite**

In [3]:
lst = geogee.load_landsat_lst(aoi, start_date='1990-01-01', end_date='1990-12-31')
# make monthly composite
monthly_lst = geogee.calculate_monthly_composite(lst, aggregate_method='mean')
monthly_lst.size().getInfo()

12

- **Export the LST collection**

In [ ]:
geesat.export_img_to_googledrive(
    ds=monthly_lst,
    aoi=aoi,
    folder_name='geesat_test',
    file_name='test',
    res=30
)

### 2. Load Sentinel-2 and mask its cloud and do some composites

In [4]:
sen2data = geogee.sen2_cloud_mask(aoi, start_date='2020-01-01', end_date='2020-12-31', cloud_filter=50)
# make monthly composite
monthly_sen2data = geogee.calculate_monthly_composite(sen2data, aggregate_method='median')
# make n-day composite 
composite = geogee.calculate_nday_composite(sen2data, aggregate_method='median', n_days=16)

### 3. Load Sentinel-1 data and calculate indices

In [6]:
sen1data = ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(aoi).filterDate('2020-01-01', '2020-12-31')
sen1indices = geogee.calculate_sen1_indices(sen1data)
print(sen1indices.first().bandNames().getInfo())

['VV', 'VH', 'RVI', 'VH_VV_Ratio']


### 4. Calculate landsat indices

In [9]:
# Calculate predefined Landsat indices from Landsat 8 and 9 data
landsat_indices = geogee.calculate_landsat_indices(aoi=aoi, start_date='2020-01-01', end_date='2020-12-31')
print(landsat_indices.first().bandNames().getInfo())

['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'NDVI', 'LST', 'NDWI', 'TVX', 'VTI']
